In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [10]:
#Cargamos el archivo SIN encabezado para ver la estructura real
#Este archivo es un reporte de tabla dinamica de Excel, no una tabla plana
raw = pd.read_excel('Ventas_por_Asesor.xlsx', header=None)
raw.head(6)

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,Ventas X Mes,12,7,8,13,6,21,17,17,23,...,37,10,NaN,NaN,178,NaN,397,NaN,236,NaN
1,NaN,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,...,2026-06-01 00:00:00,2026-07-01 00:00:00,NaN,NaN,2024,NaN,2025,NaN,2026,NaN
2,Nombre vendedor,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,811.0,Raking,Acum,Ranking,Acum,Ranking,Acum,Ranking
3,PEDRO GONZALEZ ESPINDOLA,,,1,2,,7,2,3,3,...,4,NaN,88.0,NaN,22,NaN,45,2,21,NaN
4,VICTOR MANUEL SANCHEZ ARRO,,,,,1,1,3,,2,...,3,2,88.0,NaN,11,NaN,51,NaN,26,NaN
5,MIGUEL ANGEL SONDEREGGER,,,,1,2,7,5,2,2,...,NaN,NaN,28.0,6,27,1,1,25,0,17


In [12]:
#Identificar valores nulos por columna
valores_nulos = raw.isnull().sum()
valores_nulos

0      1
1     21
2     22
3     22
4     22
5     22
6     22
7     22
8     22
9     22
10    22
11    22
12    21
13    22
14    23
15    20
16    22
17    39
18    40
19    41
20    38
21    38
22    40
23    39
24    40
25    38
26    38
27    38
28    41
29    39
30    45
31     3
32     5
33     1
34     5
35     1
36     4
37     1
38     5
dtype: int64

In [ ]:
#Identificar valores nulos en todo el dataframe
valores_nulos = raw.isnull().sum().sum()
valores_nulos

np.int64(924)

In [7]:
#Extraemos las fechas reales de la fila 1 (columnas 1 a 30) para nombrar las columnas de meses
fechas = raw.iloc[1, 1:31]
nombres_meses = [f.strftime('%Y-%m') for f in fechas]
nombres_meses[:5]

['2024-02', '2024-03', '2024-04', '2024-05', '2024-06']

In [8]:
#Construimos la lista completa de nombres de columnas
columnas = ['Nombre Vendedor'] + nombres_meses + [
    'Total 2024', 'Ranking 2024', 'Acumulado 2024',
    'Ranking 2025', 'Acumulado 2025',
    'Ranking 2026', 'Acumulado 2026', 'Ranking General'
]
len(columnas)

39

In [13]:
#Nos quedamos solo con las filas de vendedores
#Se excluyen las filas 0-2 (encabezados) y la fila 54 (Total general, que no es un vendedor)
data = raw.iloc[3:54].reset_index(drop=True)
data.columns = columnas
data.head(5)

,Nombre Vendedor,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,...,2026-06,2026-07,Total 2024,Ranking 2024,Acumulado 2024,Ranking 2025,Acumulado 2025,Ranking 2026,Acumulado 2026,Ranking General
0,PEDRO GONZALEZ ESPINDOLA,,,1,2,,7,2,3,3,...,4,NaN,88.0,NaN,22,NaN,45,2,21,NaN
1,VICTOR MANUEL SANCHEZ ARRO,,,,,1,1,3,,2,...,3,2,88.0,NaN,11,NaN,51,NaN,26,NaN
2,MIGUEL ANGEL SONDEREGGER,,,,1,2,7,5,2,2,...,NaN,NaN,28.0,6,27,1,1,25,0,17
3,JOSE ANTONIO LICONA MENDE,1,,,2,,,,3,2,...,9,1,98.0,1,10,4,35,4,53,1
4,JAIME GONZALEZ CORONA,,2,3,5,2,3,3,3,4,...,NaN,NaN,25.0,8,25,2,0,30,0,17


In [14]:
#Algunas celdas que se ven vacias en realidad tienen un espacio invisible
#no un nulo real, por lo que isnull() no las detecta. Las convertimos a NaN 
data = data.replace(r'^\s*$', np.nan, regex=True)

In [15]:
#Identificar valores nulos por columna
valores_nulos = data.isnull().sum()
valores_nulos

Nombre Vendedor     0
2024-02            46
2024-03            47
2024-04            47
2024-05            45
2024-06            47
2024-07            45
2024-08            46
2024-09            44
2024-10            41
2024-11            39
2024-12            39
2025-01            38
2025-02            40
2025-03            44
2025-04            40
2025-05            42
2025-06            38
2025-07            39
2025-08            40
2025-09            37
2025-10            37
2025-11            39
2025-12            38
2026-01            40
2026-02            37
2026-03            37
2026-04            37
2026-05            40
2026-06            38
2026-07            44
Total 2024          0
Ranking 2024        2
Acumulado 2024      0
Ranking 2025        2
Acumulado 2025      0
Ranking 2026        1
Acumulado 2026      0
Ranking General     2
dtype: int64

In [16]:
#Identificar valores nulos en todo el dataframe
valores_nulos = data.isnull().sum().sum()
valores_nulos

np.int64(1238)

In [17]:
## Metodos de sustitución de valores nulos
#Realizamos una copia del dataframe
data2 = data.copy()

In [18]:
#Primer método de sustitución de valores nulos
#En las columnas de meses, un espacio en blanco significa que ese vendedor no vendio nada ese mes
#Por lo tanto, sustituir por 0
data2[nombres_meses] = data2[nombres_meses].fillna(0)

In [19]:
#Corroboramos valores nulos
valores_nulos = data2.isnull().sum()
valores_nulos

Nombre Vendedor    0
2024-02            0
2024-03            0
2024-04            0
2024-05            0
2024-06            0
2024-07            0
2024-08            0
2024-09            0
2024-10            0
2024-11            0
2024-12            0
2025-01            0
2025-02            0
2025-03            0
2025-04            0
2025-05            0
2025-06            0
2025-07            0
2025-08            0
2025-09            0
2025-10            0
2025-11            0
2025-12            0
2026-01            0
2026-02            0
2026-03            0
2026-04            0
2026-05            0
2026-06            0
2026-07            0
Total 2024         0
Ranking 2024       2
Acumulado 2024     0
Ranking 2025       2
Acumulado 2025     0
Ranking 2026       1
Acumulado 2026     0
Ranking General    2
dtype: int64

In [20]:
#Segundo método de sustitución de valores nulos
#Las columnas de Ranking tienen muy pocos nulos 
#Un ranking vacio significa que ese vendedor no alcanzo a entrar al ranking ese periodo
#Usamos 0 porque el ranking real empieza en 1, entonces 0 significa claramente "sin ranking"
columnas_ranking = ['Ranking 2024', 'Ranking 2025', 'Ranking 2026', 'Ranking General']
data2[columnas_ranking] = data2[columnas_ranking].fillna(0)

In [21]:
#Corroboramos valores nulos en todo el dataframe
valores_nulos = data2.isnull().sum().sum()
valores_nulos

np.int64(0)

In [22]:
#Convertir DataFrame a CSV
data2.to_csv("Ventas_por_Asesor_sin_nulos.csv", index=False)